# Lab 10: Logistic Regression 1

รายวิชา `229351: Statistical Learning for Data Science 1`

Lab นี้ใช้ข้อมูล `Default` ชุดเดียวกับ ISLR บทที่ 4 และสไลด์ Lec 10 ตัวเลขทุกตัวใน Exercise 1-5 จึงตรงกับสไลด์ ถ้าค่าที่ได้ไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด ไม่ใช่เรื่องของการปัดเศษ Exercise 6 ใช้ข้อมูลจริงจาก Framingham Heart Study

Lab นี้เขียนสูตรด้วย numpy ก่อน แล้วจึงใช้ `statsmodels` และ scikit-learn เป็นตัวตรวจคำตอบ ไม่ใช่ในทางกลับกัน

## เป้าหมาย

1. เขียน sigmoid, logit และ odds ด้วย numpy
2. อธิบายด้วยตัวเลขว่าเหตุใด linear regression บน `y` ที่เป็น 0/1 จึงใช้ไม่ได้
3. fit logistic regression ด้วย `statsmodels` และอ่านตารางผลลัพธ์ได้
4. ทำนายความน่าจะเป็นด้วยมือจากสัมประสิทธิ์
5. ตีความ $\beta_1$ บนสเกล log-odds และเป็น odds ratio
6. เขียน log-likelihood ด้วยมือ และอธิบายว่าสูตรที่ใช้ $p(x_i)$ กับทุกคนผิดอย่างไร
7. อธิบาย confounding ระหว่าง `student` กับ `balance`
8. เตรียมข้อมูลจริงก่อน fit
9. หลีกเลี่ยงกับดัก `sm.Logit` ที่ไม่มี intercept และ scikit-learn ที่หยุดก่อนลู่เข้า
10. เทียบ accuracy กับ baseline และรายงานผลอย่างซื่อสัตย์

## 0. Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression

pd.set_option("display.float_format", lambda value: f"{value:.6g}")
plt.rcParams["figure.figsize"] = (9, 3.5)
plt.rcParams["axes.grid"] = True

# ข้อมูลทั้งสองไฟล์ถูก pin ไว้ที่ commit หนึ่งโดยเฉพาะ เพื่อให้คำตอบเชิงตัวเลขไม่เปลี่ยนตามเวลา
DEFAULT_URL = "https://raw.githubusercontent.com/intro-stat-learning/ISLP/a1f4e43ca88d4c9a6f186930181d628b9270015d/ISLP/data/Default.csv"
FRAMINGHAM_URL = "https://raw.githubusercontent.com/skitimo/ds351-data/6cf61a33e9eff25baef10cbff74a2856925db04a/framingham.csv"

## 1. ข้อมูล `Default` (8 คะแนน)

| คอลัมน์ | ความหมาย |
|---|---|
| `default` | ผิดนัดชำระบัตรเครดิต (`"Yes"`/`"No"`) |
| `student` | เป็นนักศึกษาไหม (`"Yes"`/`"No"`) |
| `balance` | ยอดค้างชำระเฉลี่ย (\$) |
| `income` | รายได้ต่อปี (\$) |

model ต้องการตัวเลข ไม่ใช่ข้อความ จึงต้องสร้างคอลัมน์ใหม่สามคอลัมน์

- `y`: `1` ถ้า `default == "Yes"` และ `0` ถ้าไม่ใช่ เป็น `int`
- `student_yes`: dummy ของ `student` แบบเดียวกับคาบที่ 6 baseline คือ "ไม่ใช่นักศึกษา" เป็น `int`
- `income_k`: รายได้เป็น **พันดอลลาร์** เพื่อให้สัมประสิทธิ์อ่านรู้เรื่อง (ISLR Table 4.3 ใช้หน่วยนี้)

In [ ]:
raw = pd.read_csv(DEFAULT_URL)
print(raw.shape)
display(raw.head(3))

In [ ]:
# TODO: สร้าง df ที่เป็นสำเนาของ raw และเพิ่มคอลัมน์ y, student_yes, income_k
df = raw.copy()
df["y"] = ...
df["student_yes"] = ...
df["income_k"] = ...

print(f"n = {len(df)}, defaults = {df['y'].sum()} ({df['y'].mean():.2%}), students = {df['student_yes'].sum()}")

In [ ]:
assert len(df) == 10000
assert df["y"].dtype.kind == "i" and set(df["y"].unique()) == {0, 1}
assert df["y"].sum() == 333, "should be 333 defaults"
assert df["student_yes"].sum() == 2944, "should be 2944 students"
assert abs(df["income_k"].mean() - df["income"].mean() / 1000) < 1e-9
print("data OK")

## Exercise 1: Sigmoid, logit และ odds (14 คะแนน)

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \text{odds}(p) = \frac{p}{1-p}, \qquad \text{logit}(p) = \log\frac{p}{1-p}$$

เขียนทั้งสามฟังก์ชันด้วย numpy ให้รับได้ทั้งตัวเลขเดี่ยวและ array ห้ามใช้ `scipy.special.expit` หรือ `scipy.special.logit`

In [ ]:
# TODO: เขียน sigmoid(z), odds(p) และ logit(p)
def sigmoid(z):
    ...


def odds(p):
    ...


def logit(p):
    ...

In [ ]:
assert sigmoid(0) == 0.5
assert np.allclose(sigmoid([-4, -2, 2, 4]), [0.0180, 0.1192, 0.8808, 0.9820], atol=5e-5)
z = np.linspace(-6, 6, 25)
assert np.allclose(sigmoid(-z), 1 - sigmoid(z)), "symmetry: sigma(-z) = 1 - sigma(z)"
assert np.allclose(logit(sigmoid(z)), z), "logit must undo sigmoid"
assert np.isclose(odds(0.2), 0.25) and np.isclose(odds(0.75), 3.0)
print("Exercise 1 OK")

**เขียนคำตอบข้อ 1** (ตอบให้ครบทั้งสามข้อ)

1. ทำไม $\sigma(z)$ จึงไม่มีทางเท่ากับ $0$ หรือ $1$ พอดี ไม่ว่า $z$ จะเป็นเท่าใด
2. ถ้า $p = 0.75$ odds และ log-odds เป็นเท่าใด และ odds นั้นอ่านเป็นภาษาพูดว่าอย่างไร
3. ทำไมการตั้งให้ **log-odds** เป็นเส้นตรงใน $x$ จึงใช้ได้ แต่การตั้งให้ **$p$** เป็นเส้นตรงใน $x$ ใช้ไม่ได้ อธิบายโดยพูดถึงช่วงของค่าที่แต่ละฝั่งเป็นได้

_(เขียนคำตอบที่นี่)_

## Exercise 2: Linear probability model (12 คะแนน)

fit linear regression แบบธรรมดา (OLS) ของ `y` บน `balance` แล้วดูว่าเกิดอะไรขึ้นกับค่าที่ fit ได้

In [ ]:
# TODO: fit OLS ของ y บน balance (อย่าลืม intercept) แล้วนับจำนวน fitted value ที่ติดลบ
lpm = ...
fitted = ...
n_negative = ...
zero_crossing = ...     # balance ที่เส้นตรงตัดศูนย์: -intercept / slope

print(lpm.params)
print(f"negative fitted values: {n_negative}, min {fitted.min():.4f}, max {fitted.max():.4f}")
print(f"the line crosses 0 at balance = {zero_crossing:.1f}")

In [ ]:
assert n_negative == 3123
assert round(fitted.min(), 4) == -0.0752 and round(fitted.max(), 4) == 0.2695
assert round(zero_crossing, 1) == 579.0

fig, ax = plt.subplots()
ax.scatter(df["balance"], df["y"], s=3, alpha=0.1, color="grey")
grid = np.linspace(0, df["balance"].max(), 200)
ax.plot(grid, lpm.params["const"] + lpm.params["balance"] * grid, color="darkgoldenrod", lw=2)
ax.axhline(0, color="firebrick", ls="--", lw=0.8)
ax.axhline(1, color="firebrick", ls="--", lw=0.8)
ax.set_xlabel("balance"); ax.set_ylabel("fitted 'probability'")
plt.show()

**เขียนคำตอบข้อ 2** (ตอบให้ครบทั้งสามข้อ)

1. มีลูกค้ากี่คนที่ได้ "ความน่าจะเป็น" ติดลบ และคนกลุ่มนั้นคือคนที่ `balance` เท่าใด
2. ค่าที่ fit ได้สูงสุดเป็นเท่าใด ทำไมเส้นตรงจึงขึ้นไปไม่ถึงบริเวณที่ความน่าจะเป็นสูง ทั้งที่ลูกค้าที่ยอดค้างสูงมากส่วนใหญ่ผิดนัดจริง
3. model นี้ยังมีประโยชน์อะไรเหลืออยู่บ้างไหม และใช้ทำอะไรไม่ได้เลย

_(เขียนคำตอบที่นี่)_

## Exercise 3: Logistic regression บน `balance` (18 คะแนน)

$$p(\text{balance}) = \sigma(\beta_0 + \beta_1 \cdot \text{balance})$$

ใช้ formula API ของ `statsmodels` ซึ่ง **ใส่ intercept ให้อัตโนมัติ** ผลลัพธ์ควรตรงกับ ISLR Table 4.1

In [ ]:
# TODO: fit logistic regression ของ y บน balance ด้วย smf.logit
fit1 = ...
print(fit1.summary())

In [ ]:
b0, b1 = fit1.params["Intercept"], fit1.params["balance"]
assert round(b0, 4) == -10.6513 and round(b1, 4) == 0.0055
assert round(fit1.tvalues["balance"], 2) == 24.95      # statsmodels calls the z-statistic tvalues
print(f"b0 = {b0:.4f}, b1 = {b1:.6f}")

### ทำนายด้วยมือ

ใช้ `b0`, `b1` และฟังก์ชัน `sigmoid` ของ Exercise 1 คำนวณ $\hat{p}$ ที่ `balance` = 1000, 1500, 2000, 2500 แล้วเทียบกับ `fit1.predict`

In [ ]:
# TODO: ทำนายด้วยมือ แล้วคำนวณ odds ratio ต่อ $100 และ balance ที่ทำให้ p-hat = 0.5
balances = np.array([1000, 1500, 2000, 2500])
p_hand = ...          # ใช้ sigmoid, b0, b1
p_model = fit1.predict(pd.DataFrame({"balance": balances})).to_numpy()
or_100 = ...          # odds ratio เมื่อ balance เพิ่ม 100
balance_half = ...    # balance ที่ทำให้ b0 + b1 * balance = 0

print(pd.DataFrame({"balance": balances, "p_hand": p_hand, "p_model": p_model}))
print(f"odds ratio per $100 = {or_100:.4f}, p-hat = 0.5 at balance = {balance_half:.1f}")

In [ ]:
assert np.allclose(p_hand, p_model)
assert np.allclose(p_hand, [0.00575, 0.08295, 0.58577, 0.95673], atol=5e-6)
assert round(or_100, 4) == 1.7331
assert round(balance_half, 1) == 1937.0
print("Exercise 3 OK")

**เขียนคำตอบข้อ 3** (ตอบให้ครบทั้งสี่ข้อ)

1. ตีความ $\hat{\beta}_1$ ทั้งบนสเกล log-odds (ต่อ \$1) และเป็น odds ratio (ต่อ \$100)
2. เพื่อนคนหนึ่งเขียนว่า "ยอดค้างเพิ่ม \$1 โอกาสผิดนัดเพิ่ม 0.0055" ผิดตรงไหน
3. จากตาราง `p_hand` ยอดค้างเพิ่มช่วงละ \$500 เท่ากัน แต่ $\hat{p}$ เพิ่มไม่เท่ากัน อธิบายว่าทำไม และ $\hat{p}$ เปลี่ยนเร็วที่สุดที่ `balance` เท่าใด
4. $z$-statistic ของ `balance` ทดสอบสมมติฐานอะไร และผลบอกอะไร

_(เขียนคำตอบที่นี่)_

## Exercise 4: Log-likelihood ด้วยมือ (14 คะแนน)

$$\log \ell(\beta_0, \beta_1) = \sum_{i=1}^{n} \bigl[ y_i\, \eta_i - \log(1 + e^{\eta_i}) \bigr], \qquad \eta_i = \beta_0 + \beta_1 x_i$$

เป็นรูปเดียวกับ $\sum [y_i \log p_i + (1-y_i) \log(1-p_i)]$ ในสไลด์ แต่ไม่ต้องคำนวณ $\log$ ของค่าที่ใกล้ศูนย์ ใช้ `np.log1p` สำหรับ $\log(1 + \cdot)$

In [ ]:
# TODO: เขียน loglik(b0, b1, x, y) ตามสูตรข้างบน
def loglik(b0, b1, x, y):
    ...


x_bal, y_def = df["balance"].to_numpy(), df["y"].to_numpy()
ll_hat = loglik(b0, b1, x_bal, y_def)
print(f"by hand {ll_hat:.2f}   statsmodels {fit1.llf:.2f}   null model {fit1.llnull:.2f}")

In [ ]:
assert abs(ll_hat - fit1.llf) < 1e-6
assert round(ll_hat, 2) == -798.23

# สี่คู่ (b0, b1) จากตารางบนสไลด์ "Log-likelihood มีจุดสูงสุด"
candidates = [(-8.2991, 0.004), (-9.8540, 0.005), (-11.4631, 0.006), (-13.1088, 0.007)]
expected = [-826.72, -800.97, -800.65, -817.52]
for (c0, c1), want in zip(candidates, expected):
    got = loglik(c0, c1, x_bal, y_def)
    assert abs(got - want) < 0.05 and got < ll_hat
    print(f"b1 = {c1:.3f}  log L = {got:.2f}")
print("Exercise 4 OK: every candidate is below the MLE")

### สูตรที่ผิด

เอกสารบางฉบับเขียน likelihood เป็น $L = \prod_i p(x_i)$ คือใช้ $p(x_i)$ กับ **ทุกคน** เขียน `wrong_loglik` ตามสูตรนั้น แล้วลองเพิ่ม intercept ทีละ 5

In [ ]:
# TODO: wrong_loglik(b0, b1, x) = sum log sigma(b0 + b1 x) -- ไม่มี y อยู่ในสูตรเลย
def wrong_loglik(b0, b1, x):
    ...


for shift in (0, 5, 10, 20):
    print(f"b0 + {shift:2d}: wrong {wrong_loglik(b0 + shift, b1, x_bal):12.2f}   "
          f"right {loglik(b0 + shift, b1, x_bal, y_def):12.2f}")

In [ ]:
vals = [wrong_loglik(b0 + s, b1, x_bal) for s in (0, 5, 10, 20)]
assert all(a < b for a, b in zip(vals, vals[1:])), "the wrong likelihood only ever goes up"
assert loglik(b0 + 5, b1, x_bal, y_def) < ll_hat
print("the wrong formula rewards predicting 'default' for everyone")

**เขียนคำตอบข้อ 4** (ตอบให้ครบทั้งสามข้อ)

1. ทำไมสูตรที่ถูกต้องจึงมีเลขชี้กำลัง $y_i$ และ $1 - y_i$ อธิบายว่าแต่ละตัวทำหน้าที่อะไร
2. `wrong_loglik` ผิดตรงไหน ถ้าส่งให้ optimiser หาค่าที่มากที่สุด จะได้ model แบบใด อ้างอิงตารางที่พิมพ์ออกมา
3. `fit1.llf` เทียบกับ `fit1.llnull` (model ที่มีแต่ intercept) บอกอะไรเกี่ยวกับ `balance`

_(เขียนคำตอบที่นี่)_

## Exercise 5: Multiple logistic regression และ confounding (16 คะแนน)

fit สอง model แล้วเทียบสัมประสิทธิ์ของ `student_yes`

1. `y ~ student_yes` (ISLR Table 4.2)
2. `y ~ balance + income_k + student_yes` (ISLR Table 4.3)

In [ ]:
# TODO: fit สอง model และคำนวณสัดส่วนผิดนัด กับ balance เฉลี่ย แยกตาม student_yes
fit_s = ...
fit_all = ...
rate_by_student = ...       # Series ที่ index คือ 0/1
balance_by_student = ...    # Series ที่ index คือ 0/1

print(fit_s.params, fit_all.params, sep="\n\n")
print(rate_by_student, balance_by_student, sep="\n\n")

In [ ]:
assert round(fit_s.params["student_yes"], 4) == 0.4049
assert round(fit_all.params["student_yes"], 4) == -0.6468
assert round(fit_all.params["income_k"], 4) == 0.0030
assert round(fit_all.pvalues["income_k"], 4) == 0.7115
assert round(rate_by_student[1], 4) == 0.0431 and round(rate_by_student[0], 4) == 0.0292
assert round(balance_by_student[1], 1) == 987.8 and round(balance_by_student[0], 1) == 771.8
print("Exercise 5 OK")

In [ ]:
# ทำนายนักศึกษาและไม่ใช่นักศึกษาที่ balance 1500, income $40k
same_customer = pd.DataFrame({"balance": [1500, 1500], "income_k": [40, 40], "student_yes": [1, 0]})
print(fit_all.predict(same_customer).round(4).tolist())     # [student, not student]

# default rate by balance bin, per group -- the left panel of the confounding slide
bins = pd.cut(df["balance"], np.arange(0, 2700, 250), include_lowest=True)
table = df.groupby([bins, "student_yes"], observed=True)["y"].mean().unstack()
mids = [interval.mid for interval in table.index]
fig, ax = plt.subplots()
ax.plot(mids, table[1], "o-", label="student")
ax.plot(mids, table[0], "s-", label="not student")
ax.set_xlabel("balance (bins of 250)"); ax.set_ylabel("default rate"); ax.legend()
plt.show()

**เขียนคำตอบข้อ 5** (ตอบให้ครบทั้งสามข้อ)

1. อธิบายด้วยตัวเลขที่คำนวณได้ว่าทำไมสัมประสิทธิ์ของ `student_yes` จึงเป็นบวกเมื่ออยู่ตัวเดียว แต่เป็นลบเมื่อมี `balance` อยู่ด้วย
2. ธนาคารสองแห่ง แห่งหนึ่งไม่รู้ยอดค้างของลูกค้า อีกแห่งรู้ ควรมองนักศึกษาว่าเสี่ยงกว่าหรือเสี่ยงน้อยกว่าคนอื่น
3. `income_k` มีนัยสำคัญไหม ถ้าเปลี่ยนหน่วยกลับเป็นดอลลาร์ สัมประสิทธิ์ $z$ และ $p$-value จะเปลี่ยนอย่างไร

_(เขียนคำตอบที่นี่)_

## Exercise 6: Framingham การเตรียมข้อมูล และสองกับดัก (18 คะแนน)

target คือ `TenYearCHD` (1 = เป็นโรคหลอดเลือดหัวใจภายใน 10 ปี) และใช้ predictor ทั้ง 15 ตัว

### 6.1 ค่าที่หายไป

In [ ]:
fram = pd.read_csv(FRAMINGHAM_URL)
print(fram.shape)
print(fram.isna().sum()[fram.isna().sum() > 0].sort_values(ascending=False))

In [ ]:
# TODO: ตัดแถวที่มีค่าหาย แยก y_f กับ X_f แล้ว standardize X_f เป็น Xs
#       และคำนวณสัดส่วน CHD ในแถวที่ถูกตัดทิ้ง
complete = ...
dropped = ...       # แถวที่มีค่าหายอย่างน้อยหนึ่งค่า
y_f = ...
X_f = ...           # predictor ทั้ง 15 ตัว
Xs = ...            # (X_f - mean) / sd

print(f"kept {len(complete)}, dropped {len(dropped)}")
print(f"CHD rate: kept {y_f.mean():.4f}, dropped {dropped['TenYearCHD'].mean():.4f}")

In [ ]:
assert len(complete) == 3658 and len(dropped) == 582
assert int(y_f.sum()) == 557 and X_f.shape[1] == 15
assert np.allclose(Xs.mean(), 0, atol=1e-10) and np.allclose(Xs.std(), 1)
assert round(dropped["TenYearCHD"].mean(), 4) == 0.1495
print("6.1 OK")

### 6.2 กับดักที่ 1: `sm.Logit` ไม่ใส่ intercept ให้

เซลล์ข้างล่างรัน model ที่ **ผิด** โดยตั้งใจ ดูผลลัพธ์ก่อน แล้วเขียน model ที่ถูกในเซลล์ถัดไป

In [ ]:
bad = sm.Logit(y_f, Xs).fit(disp=0)          # the array API: no intercept is added
p_bad = bad.predict(Xs)
print(f"log L = {bad.llf:.2f}   mean p-hat = {p_bad.mean():.4f}   actual rate = {y_f.mean():.4f}")
print(f"people predicted CHD at 0.5: {int((p_bad > 0.5).sum())}")

In [ ]:
# TODO: fit model ที่ถูก (มี intercept) เป็น good และคำนวณ p_good
good = ...
p_good = ...
print(f"log L = {good.llf:.2f}   mean p-hat = {p_good.mean():.4f}   actual rate = {y_f.mean():.4f}")

In [ ]:
assert round(bad.llf, 2) == -2436.61 and round(p_bad.mean(), 4) == 0.4987
assert round(good.llf, 2) == -1377.24
assert abs(p_good.mean() - y_f.mean()) < 1e-6, "with an intercept, mean p-hat equals the rate of y = 1"
print("6.2 OK")

### 6.3 กับดักที่ 2: scikit-learn ที่หยุดก่อนลู่เข้า

เซลล์ข้างล่างใช้ค่าเริ่มต้นของ `LogisticRegression()` บนข้อมูลที่ **ไม่ได้** standardize แล้วเก็บ warning ที่เกิดขึ้นไว้ให้ดู

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sk_default = LogisticRegression().fit(X_f, y_f)
print(sorted({type(w.message).__name__ for w in caught}), "iterations:", sk_default.n_iter_[0])

raw_sm = sm.Logit(y_f, sm.add_constant(X_f)).fit(disp=0)
compare = pd.DataFrame({"statsmodels": raw_sm.params[X_f.columns],
                        "sklearn default": sk_default.coef_[0]}, index=X_f.columns)
print(compare.loc[["age", "male", "sysBP", "glucose"]].round(4))

In [ ]:
# TODO: fit LogisticRegression แบบไม่มี penalty บน Xs (ข้อมูลที่ standardize แล้ว)
#       แล้ววัดว่าสัมประสิทธิ์ห่างจากของ good มากที่สุดเท่าใด
sk = ...
gap = ...
print(f"max |sklearn - statsmodels| = {gap:.4f}")

In [ ]:
assert "ConvergenceWarning" in {type(w.message).__name__ for w in caught}
assert round(compare.loc["age", "sklearn default"], 4) == 0.0252
assert gap < 0.01
print("6.3 OK")

### 6.4 จากความน่าจะเป็นสู่การจำแนก

ตัดที่ 0.5 แล้วเทียบกับการทายว่า "ไม่เป็นโรค" ทุกคน

In [ ]:
# TODO: จำแนกด้วยเกณฑ์ 0.5 จาก p_good แล้วคำนวณ accuracy, baseline และจำนวนผู้ป่วยที่จับได้
y_hat = ...
accuracy = ...
baseline = ...          # accuracy ของการทายว่า 0 ทุกคน
caught_cases = ...      # y_hat = 1 และ y_f = 1

print(f"accuracy {accuracy:.4f}   always 'no CHD' {baseline:.4f}")
print(f"predicted CHD {int(y_hat.sum())}, caught {caught_cases} of {int(y_f.sum())} real cases")

In [ ]:
assert round(accuracy, 4) == 0.8565 and round(baseline, 4) == 0.8477
assert caught_cases == 51 and int(y_hat.sum()) == 70
print("6.4 OK")

**เขียนคำตอบข้อ 6** (ตอบให้ครบทั้งสี่ข้อ)

1. ทำไม model ที่ไม่มี intercept จึงทายค่าเฉลี่ยของ $\hat{p}$ ใกล้ 0.5 บนข้อมูลที่ standardize แล้ว และมีวิธีตรวจง่าย ๆ อย่างไรว่า model ของเรามี intercept
2. ทำไมจึงไม่ควรปิด `ConvergenceWarning` ทิ้ง และควรแก้อย่างไร
3. model มี accuracy $0.8565$ ดีไหม เทียบกับ baseline และจำนวนผู้ป่วยที่จับได้ แล้วตอบว่าถ้าใช้คัดกรองผู้ป่วยจริงจะเกิดอะไรขึ้น
4. ถ้าจะประเมิน model นี้ให้ดีกว่า accuracy บน training set ควรวัดอะไรเพิ่ม

_(เขียนคำตอบที่นี่)_

## โจทย์เสริม (ไม่คิดคะแนน)

1. fit `TenYearCHD ~ age` บนข้อมูลดิบ (ไม่ standardize) แล้วคำนวณ odds ratio ต่อ 10 ปี บนสไลด์ model ที่มีตัวแปรครบ 15 ตัวได้ $1.8873$ ของคุณได้เท่าใด และทำไมจึงต่างกัน
2. เปลี่ยนเกณฑ์จาก 0.5 เป็น 0.2 แล้วนับใหม่ว่าจับผู้ป่วยได้กี่คน และทายผิดว่าป่วยกี่คน
3. สุ่มแถวของ Framingham มา 100 แถว (`random_state=0`) standardize แล้วเทียบสัมประสิทธิ์ `age` ระหว่าง `LogisticRegression()` กับ `LogisticRegression(C=np.inf)` บนสไลด์ได้ $1.082$ กับ $1.718$

## Checklist ก่อนส่ง

- [ ] โหลด `Default` จาก URL ที่ pin ไว้ได้ 10,000 แถว ผิดนัด 333 คน นักศึกษา 2,944 คน
- [ ] `sigmoid`, `odds` และ `logit` ผ่านทุก assert
- [ ] linear probability model ให้ค่าติดลบ 3,123 คน และเส้นตัดศูนย์ที่ `balance` 579.0
- [ ] `fit1` ได้ $-10.6513$ และ $0.0055$ และ $z = 24.95$
- [ ] `p_hand` ตรงกับ `fit1.predict` odds ratio ต่อ \$100 = 1.7331 และ $\hat{p} = 0.5$ ที่ 1937.0
- [ ] `loglik` ได้ $-798.23$ และทุกคู่ในตารางได้ค่าต่ำกว่า
- [ ] `student_yes` ได้ $+0.4049$ ตัวเดียว และ $-0.6468$ ใน model เต็ม
- [ ] Framingham เหลือ 3,658 แถว positive 557 คน
- [ ] model ที่มี intercept ได้ค่าเฉลี่ย $\hat{p}$ = 0.1523 เท่ากับสัดส่วนจริง
- [ ] `LogisticRegression(C=np.inf)` ห่างจาก `statsmodels` ไม่เกิน 0.01
- [ ] accuracy 0.8565 baseline 0.8477 จับผู้ป่วยได้ 51 คน
- [ ] ตอบคำถามทุกข้อในทุก markdown cell ที่ขึ้นต้นด้วย "เขียนคำตอบ"
- [ ] restart kernel แล้ว run all ผ่านตลอดโดยไม่มี error
- [ ] เปลี่ยนชื่อไฟล์เป็น `lab10_studentid.ipynb` ก่อนส่ง